In [7]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate_hkqai_done").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict_ccpvdz"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099

        data_error_scf_ele = data["error_scf_ele"].to_numpy()
        data_error_dft_ele = data["error_dft_ele"].to_numpy()
        data_error_scf_dip = data["error_scf_dip"].to_numpy()
        data_error_dft_dip = data["error_dft_dip"].to_numpy()
        data_subset[f"{data_path_name}_summary"] = {
            "error_scf_ele": data_error_scf_ele,
            "error_dft_ele": data_error_dft_ele,
            "error_scf_dip": data_error_scf_dip,
            "error_dft_dip": data_error_dft_dip,
        }

        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if "modified_dft_d3bj" in data.columns:
            data_dft_d3bj = data["modified_dft_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3bj = data_d3bj
        if "modified_dft_d3zero" in data.columns:
            data_dft_d3zero = data["modified_dft_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3zero = data_d3zero
        if "modified_ai_d3bj" in data.columns:
            data_ai_d3bj = data["modified_ai_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3bj = data_d3bj
        if "modified_ai_d3zero" in data.columns:
            data_ai_d3zero = data["modified_ai_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
                "error_scf_ele": [],
                "error_dft_ele": [],
                "error_scf_dip": [],
                "error_dft_dip": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(
                            f"Warning: {i_molecule_name} not found in {data_path.stem} data file"
                        )
                    continue
                data_subset[name_subset]["error_scf_ele"].append(
                    data_error_scf_ele[col[0]]
                )
                data_subset[name_subset]["error_dft_ele"].append(
                    data_error_dft_ele[col[0]]
                )
                data_subset[name_subset]["error_scf_dip"].append(
                    data_error_scf_dip[col[0]]
                )
                data_subset[name_subset]["error_dft_dip"].append(
                    data_error_dft_dip[col[0]]
                )

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_ai = np.argsort(data_subset[name_subset]["ai"])[
                    ::-1
                ][:5]
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ][:5]
                for i in range(len(argsort_atomic_energy_ai)):
                    i_reaction_name = data_subset[name_subset]["name"][
                        argsort_atomic_energy_ai[i]
                    ]
                    i_reaction = json_data[f"reaction-{i_subset}"][i_reaction_name]
                    print(
                        f"Top {i+1} AI: {data_subset[name_subset]['ai'][argsort_atomic_energy_ai[i]]} kcal/mol, {i_reaction_name} in {name_subset}",
                    )
                    systems_list = i_reaction["systems"]
                    stoichiometry_list = i_reaction["stoichiometry"]
                    for j in range(len(systems_list)):
                        mole_name = (
                            systems_list[j]
                            if i_subset == "BH76RC"
                            else f"{i_subset}-{systems_list[j]}"
                        )
                        stoichiometry = int(stoichiometry_list[j])
                        if mole_name in json_data:
                            if isinstance(json_data[mole_name], str):
                                mole_name = json_data[mole_name]
                        print(f"  {stoichiometry} * {mole_name}", end="")
                        col = np.where(data_name == mole_name)[0]
                        if col.size == 1:
                            error_energy_ai = data_scf[col[0]] - data_cc[col[0]]
                            print(f"  {stoichiometry} * {error_energy_ai}", end="")
                    print()

                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

df_summary_subset_ele = pd.DataFrame(
    columns=pd.MultiIndex.from_product(
        [
            data_path_name_list,
            [
                "error_scf_ele",
                "error_dft_ele",
                "error_scf_dip",
                "error_dft_dip",
            ],
        ],
        names=["data_path", "Ele type"],
    )
)

for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_dip"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_dip"]
    )

    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}

        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_ele")] = (
                np.mean(data_subset[name_subset]["error_scf_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_ele")] = (
                np.mean(data_subset[name_subset]["error_dft_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_dip")] = (
                np.mean(data_subset[name_subset]["error_scf_dip"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_dip")] = (
                np.mean(data_subset[name_subset]["error_dft_dip"])
            )

            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    "DONE"
                    if (
                        (
                            len(data_subset[name_subset]["ai"])
                            == len(data_subset[name_subset]["name"])
                        )
                        and (len(data_subset[name_subset]["ai"]) != 0)
                    )
                    else f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(
        mean_absolute_deviation_list
    )
    print(
        f"Mean absolute deviation for {data_path_name}: {mean_absolute_deviation:.4f} kcal/mol"
    )
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

    wtmad_1_subset.loc["summary", (data_path_name, "Processed")] = "--"
    wtmad_2_subset.loc["summary", (data_path_name, "Processed")] = "--"
    for d3_name in ["", "_d3bj", "_d3zero"]:
        wtmad_1_subset.loc["summary", (data_path_name, f"AI{d3_name.upper()}")] = 0
        wtmad_1_subset.loc["summary", (data_path_name, f"DFT{d3_name.upper()}")] = 0
        wtmad_2_subset.loc["summary", (data_path_name, f"AI{d3_name.upper()}")] = 0
        wtmad_2_subset.loc["summary", (data_path_name, f"DFT{d3_name.upper()}")] = 0
        for name_set in full_subset_dict.keys():
            wtmad_1_subset.loc[
                "summary", (data_path_name, f"AI{d3_name.upper()}")
            ] += wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            wtmad_1_subset.loc[
                "summary", (data_path_name, f"DFT{d3_name.upper()}")
            ] += wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")]
            wtmad_2_subset.loc[
                "summary", (data_path_name, f"AI{d3_name.upper()}")
            ] += wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            wtmad_2_subset.loc[
                "summary", (data_path_name, f"DFT{d3_name.upper()}")
            ] += wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")]

# print("Summary")
# display(df_summary_subset_ele)
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# save summary to csv with date
df_summary_subset_ele.to_csv(f"../validate/df_summary_subset_ele_{date}.csv")
df_summary_subset.to_csv(f"../validate/summary_subset_{date}.csv")
mean_subset.to_csv(f"../validate/mean_subset_{date}.csv")
wtmad_1_subset.to_csv(f"../validate/wtmad_1_subset_{date}.csv")
wtmad_2_subset.to_csv(f"../validate/wtmad_2_subset_{date}.csv")
# save summary to excel with date
df_summary_subset_ele.to_excel(f"../validate/df_summary_subset_ele_{date}.xlsx")
df_summary_subset.to_excel(f"../validate/summary_subset_{date}.xlsx")
mean_subset.to_excel(f"../validate/mean_subset_{date}.xlsx")
wtmad_1_subset.to_excel(f"../validate/wtmad_1_subset_{date}.xlsx")
wtmad_2_subset.to_excel(f"../validate/wtmad_2_subset_{date}.xlsx")

cc-pVDZ
Top 1 AI: 12.609903921023943 kcal/mol, 116 in 1424849_W4_11
  -1 * W4_11-p4  -1 * -13.835936387768015  4 * W4_11-p  4 * -0.3065081166860182
Top 2 AI: 5.341987040214008 kcal/mol, 136 in 1424849_W4_11
  -1 * W4_11-foof  -1 * -4.328057378443191  2 * W4_11-f  2 * 0.6256503599724965  2 * W4_11-o  2 * -0.11868552908708807
Top 3 AI: 5.219463497924153 kcal/mol, 46 in 1424849_W4_11
  -1 * W4_11-alf3  -1 * -3.2680325650726445  1 * W4_11-al  1 * 0.07447985294857062  3 * W4_11-f  3 * 0.6256503599724965
Top 4 AI: 4.821493316645501 kcal/mol, 135 in 1424849_W4_11
  -1 * W4_11-cloo  -1 * -4.751233361486811  1 * W4_11-cl  1 * 0.30763101333286613  2 * W4_11-o  2 * -0.11868552908708807
Top 5 AI: 4.2865408250218024 kcal/mol, 63 in 1424849_W4_11
  -1 * W4_11-sif  -1 * -3.7919159706216305  1 * W4_11-si  1 * -0.13102550557232462  1 * W4_11-f  1 * 0.6256503599724965
Top 1 DFT: 64.9391765203327 kcal/mol
Top 2 DFT: 64.82811637483246 kcal/mol
Top 3 DFT: 58.7866408124537 kcal/mol
Top 4 DFT: 55.61089226667

data_path   1424849                                                      \
Disp type        AI        DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       2.002477  13.710857  1.935526   13.7889  1.859815  13.685393   
sub2       5.510109   6.612164  5.412513  6.359139  5.458901   6.408048   
sub3       2.617318   6.261376  2.816465  6.625846  2.938387   6.678362   
sub4        1.93524   3.272197  1.474342  3.440323  1.710948   3.742517   
sub5       1.626727   1.321288  1.369759  0.815469  1.357554   0.809664   

data_path             1513512                       ...                       \
Disp type Processed        AI        DFT   AI_D3BJ  ... AI_D3ZERO DFT_D3ZERO   
sub1           DONE  6.282354  13.717062  6.368076  ...  6.285388  13.691463   
sub2           DONE  7.536765   6.612828  8.231324  ...  8.368537     6.4095   
sub3           DONE  4.620451    6.26131  5.023311  ...   5.06199   6.678171   
sub4           DONE   2.38953   3.272965  2.422374  ...  2.681673   3.740626   
sub5           DONE  1.250183   1.322771  0.678013  ...  0.664039   0.810875   

data_path             3036943                                            \
Disp type Processed        AI        DFT   AI_D3BJ   DFT_D3BJ AI_D3ZERO   
sub1           DONE  5.772845  13.710857  5.904607  14.259701  5.573973   
sub2           DONE  7.919127   6.612128     6.749   9.130035  5.957076   
sub3           DONE  4.636265   6.261376  5.246788   7.355579  4.851987   
sub4           DONE   3.14773   3.272197  4.543588    5.03074  4.521658   
sub5           DONE  1.175371   1.177766  0.751182   0.957237  0.718836   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1       13.713797      DONE  
sub2        6.757695      DONE  
sub3        6.934991      DONE  
sub4        4.988172      DONE  
sub5        0.899379     7 / 8  

[5 rows x 21 columns]

wtmad_1


data_path    1424849                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1        3.239113   7.420922   2.966428   7.224551   2.815004   7.166359   
sub2       13.142366   12.53052  12.443324  11.377537  12.073631  10.891896   
sub3        4.832263   6.984991   5.052323    7.34316   5.207324    7.32152   
sub4       11.556831  11.029773   7.478943   7.022789   7.675787   7.720437   
sub5       15.335367  11.571826  12.722907   6.873743  12.540889   6.757453   
summary    48.105939  49.538032  40.663925   39.84178  40.312635  39.857666   

data_path              1513512                        ...             \
Disp type Processed         AI        DFT    AI_D3BJ  ...  AI_D3ZERO   
sub1           DONE   5.917804   7.427913   5.782796  ...   5.728434   
sub2           DONE   9.677109  12.530346   8.502999  ...   8.154213   
sub3           DONE   5.362819   6.982875   5.766257  ...   5.725648   
sub4           DONE  10.472747  11.046974   5.704214  ...   6.148839   
sub5           DONE  10.880025  11.599184   5.605527  ...   5.429641   
summary          --  42.310504  49.587291  31.361794  ...  31.186775   

data_path                         3036943                                   \
Disp type DFT_D3ZERO Processed         AI        DFT    AI_D3BJ   DFT_D3BJ   
sub1        7.173561      DONE   6.543701   7.420922   5.841371   7.414578   
sub2       10.893999      DONE  11.307039  12.530489   8.468475   9.968131   
sub3        7.319476      DONE   5.546066   6.984991   6.136313   8.150912   
sub4        7.745878      DONE  10.448711  11.029773  12.102006   14.87552   
sub5        6.782008      DONE  11.006618  10.752409   6.506317   7.954853   
summary    39.914922        --  44.852136  48.718585  39.054481  48.363993   

data_path                                  
Disp type  AI_D3ZERO DFT_D3ZERO Processed  
sub1        5.796977   7.102094      DONE  
sub2        8.608195  10.005264      DONE  
sub3        5.571336   7.521852      DONE  
sub4       11.099232  13.798614      DONE  
sub5        6.133091   7.451573     7 / 8  
summary    37.208831  45.879399        --  

[6 rows x 21 columns]

wtmad_2


data_path    1424849                                                        \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1        1.438234   4.040696   1.347877   4.009887  1.298132   3.993215   
sub2        3.339073   3.635331    3.00695   3.050633  2.887856   2.955369   
sub3        1.303233   2.611064   1.379492   2.754823  1.432794   2.775157   
sub4        4.683102   4.159493   3.352776   3.139343  3.830024   3.673649   
sub5        6.331811    4.66236   5.509119   2.969375  5.489474   2.874701   
summary    17.095452  19.108945  14.596213  15.924061  14.93828   16.27209   

data_path              1513512                        ...             \
Disp type Processed         AI        DFT    AI_D3BJ  ...  AI_D3ZERO   
sub1           DONE   2.668675   4.045306   2.649786  ...   2.637463   
sub2           DONE   3.206768   3.637148   2.603922  ...   2.543548   
sub3           DONE   1.989474   2.610843   2.145594  ...       2.16   
sub4           DONE   4.130245    4.16366   2.770651  ...   3.198557   
sub5           DONE   4.368745   4.670544   2.432243  ...   2.292671   
summary          --  16.363907  19.127501  12.602197  ...  12.832239   

data_path                         3036943                                   \
Disp type DFT_D3ZERO Processed         AI        DFT    AI_D3BJ   DFT_D3BJ   
sub1        3.997743      DONE   3.026285   4.281128   2.858798   4.356088   
sub2        2.958326      DONE   3.572836   3.851611    2.11247   2.517388   
sub3        2.774852      DONE   2.164366   2.766429   2.424001   3.220901   
sub4        3.673564      DONE   4.359528   4.406994   6.311912   7.264515   
sub5        2.878168      DONE   4.106082   4.093191   2.599534   3.419076   
summary    16.282653        --  17.229098  19.399353  16.306715  20.777968   

data_path                                  
Disp type  AI_D3ZERO DFT_D3ZERO Processed  
sub1        2.797576   4.227171      DONE  
sub2        2.349116   2.697132      DONE  
sub3        2.251778   3.044636      DONE  
sub4        6.067582   7.014098      DONE  
sub5        2.397851   3.104579     7 / 8  
summary    15.863904  20.087616        --  

[6 rows x 21 columns]

Summary of Subset
MAE


data_path    1424849                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
W4_11       1.028249  29.506863   1.223261  29.989789   1.151098  29.747048   
G21EA       0.816732   9.754522   0.816599    9.75468   0.820867   9.756932   
G21IP       0.694714   8.953334    0.69616   8.951545    0.68818   8.952599   
DIPCS10     2.042735  12.310852   2.039936   12.31518   2.045245  12.283687   
PA26        3.009087   2.200766   2.985201   2.058032   2.957953   2.055317   
SIE4x4      3.430805  21.908516   3.542007  22.097166   3.562974  22.161897   
ALKBDE10    0.959132  18.125246   0.989582  18.277559   0.952777  18.132767   
YBDE18      5.939045   8.145393   5.216198   7.828353   4.841699   7.695204   
AL2X6       6.140013   5.659145   4.464174   3.449937   3.988804   3.300348   
HEAVYSB11   2.024005   5.391476   1.791244   5.463331   1.721678   5.379671   
NBPRC       3.058861    2.23255   2.703933   1.870626   2.562871   2.028757   
ALK8        4.403777   4.400063   3.334795   3.350254   2.788735   2.953216   
RC21        2.456062   4.820926   1.923738    5.39421    1.82928   5.525602   
G2RC        1.887093   5.917203   1.969408   6.249306   1.995017   6.227185   
BH76RC       0.85212   3.484561   0.891385    3.50012   0.922322   3.516546   
FH51        2.464205   3.703657   2.169993   3.443826   2.129573   3.347368   
TAUT15      1.702465    2.16453    1.66463   2.155289   1.600571   2.170609   
DC13        6.170057  13.090861   6.241887  12.478213   5.863507  11.959009   
MB16_43     13.39941  15.461604  15.297752  18.175348  16.246698  18.708523   
DARC        7.823359   10.75415   5.637199   7.786582   4.993859   7.610198   
RSE43        3.11738     3.1576   3.042746   3.052617   2.897426   2.895709   
BSR36        4.98276    8.40118   3.199812   5.170108   2.849067   5.230489   
CDIE20      1.450234   1.599507   1.451078   1.511089   1.479492   1.421578   
ISO34       2.020035   2.001743   1.914195   1.837375   1.829508   1.772453   
PArel       3.015433   1.743933   2.988405    1.73941   2.944914    1.65864   
BH76        2.803583   9.107946   2.865363   9.384703   2.949706   9.471921   
BHPERI      2.657111   3.268992   3.650976    4.91131   4.133776   5.201979   
BHDIV10     4.741552   6.277706   4.837814   6.576988   4.819558   6.463527   
INV24       3.393792   2.697333    3.50674   2.430052   3.504461   2.367796   
BHROT27     1.713103   0.853379   1.717399   0.843049   1.745492   0.815488   
PX13        1.059156  11.042023   1.034643  11.281052   1.127794  11.125104   
WCPT18      2.039614   7.967151    2.29673    8.38753   2.461054   8.465437   
RG18         0.30936   0.230018   0.307671   0.272271   0.438413   0.363283   
ADIM6       3.598266   3.059029   1.159053   0.069974   1.006802   0.047161   
S22         2.728845   2.291252   2.115574   1.027824   2.108266   1.185279   
S66         2.326488   1.894709    1.01562   0.729648   0.940707   0.863238   
WATER27      3.02651  16.299413   2.824412  20.709459   4.382597  22.096474   
CARBHB12    0.484944   1.632408   0.473346   2.236748   0.556368   2.356687   
PNICO23     0.840943   0.776429   0.940904   1.237735    0.98267   1.220119   
HAL59       2.035403   1.649786   1.813837   1.431131   1.991528   1.687656   
AHB21       1.666768   2.453821   1.835619   2.836357   1.986367   2.982124   
CHB6        1.766454    1.80532   1.689088   2.042249   1.499007   2.186047   
IL16        1.467889   1.021067   2.011731   2.478095   2.624917    3.12773   
IDISP      13.356085  13.125475   9.748251   6.829682   9.297146   6.724485   
ICONF       0.946016   0.413804   0.923008   0.398194   0.896902   0.460113   
ACONF       3.016487   0.546621   2.780804   0.134491    2.73024   0.112191   
Amino20x4   1.243276   0.656235   1.214437   0.533098   1.214533   0.545417   
PCONF21      2.09681   3.214782   1.225373   1.470386   1.250642   1.109308   
MCONF       1.596087    1.95602   1.061923   0.627

In [5]:
16.363907 / 19.127501, 12.60223 / 15.936448, 14.245033 / 17.35668

(0.8555172471301924, 0.7907803545683455, 0.8207233756686185)